# DIAGNOSTIKA — prazne tumorske RNA celice

Najdba (prek data_rna_sample.pkl): tumorske RNA celice so skoraj prazne (mediana ~2 counta),
blood celice normalne (~4575). To sega v notebook 01. Ta notebook LOKALIZIRA, kje se counti izgubijo:

- KORAK A: preveri surove .h5 tumorske datoteke (so ze prazne ob branju?)
- KORAK B: preveri anndata.concat(join='inner') (presek genov izloci tumorske gene?)
- KORAK C: preveri barcode lookup (napacna povezava celica<->metadata?)

Pozeni na Colabu (kjer so .h5). Prilepi izpise.


In [ ]:
!pip install -q --upgrade numpy scanpy scipy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob, numpy as np, pandas as pd
import scanpy as sc, anndata
RAW_DIR = '/content/drive/MyDrive/Diploma/data/GSE200996_RAW'
print('tumor .h5 datoteke:')
for f in sorted(glob.glob(os.path.join(RAW_DIR, '*_GEX_sc_tumor.h5'))):
    print(' ', os.path.basename(f))

## KORAK A — so surove tumorske .h5 ze prazne ob branju?

Preberi eno tumorsko .h5 DIREKTNO, brez filtriranja/concat. Poglej vsoto countov na celico.

In [ ]:
# vzemi eno tumorsko .h5 (npr. tisto s pacientom 24 -> P25 v imenih? preveri imena)
tumor_files = sorted(glob.glob(os.path.join(RAW_DIR, '*_GEX_sc_tumor.h5')))
f = tumor_files[0]
print('Berem:', os.path.basename(f))
ad = sc.read_10x_h5(f)
ad.var_names_make_unique()
print('oblika (celice x geni):', ad.shape)
X = ad.X
sums = np.asarray(X.sum(axis=1)).ravel()
print('vsota countov/celico: min={:.0f} max={:.0f} mediana={:.0f}'.format(sums.min(), sums.max(), np.median(sums)))
print('celic z vsoto=0:', int((sums==0).sum()), '/', len(sums))
print('celic z vsoto>100:', int((sums>100).sum()), '/', len(sums))
print('-> ce so ze tu prazne: problem je .h5 sam (branje/format).')
print('-> ce so tu POLNE: problem je kasneje (concat/filter/lookup).')
print('var_names primer:', list(ad.var_names[:5]))

## KORAK A2 — primerjaj var_names blood vs tumor .h5

Ce imata blood in tumor .h5 razlicna gene imena, join='inner' izprazni presek.

In [ ]:
blood_files = sorted(glob.glob(os.path.join(RAW_DIR, '*_GEX_sc_PBMC.h5')))
adb = sc.read_10x_h5(blood_files[0]); adb.var_names_make_unique()
adt = sc.read_10x_h5(tumor_files[0]); adt.var_names_make_unique()
gb, gt = set(adb.var_names), set(adt.var_names)
print('blood genov:', len(gb), '| tumor genov:', len(gt))
print('presek:', len(gb & gt), '| samo blood:', len(gb - gt), '| samo tumor:', len(gt - gb))
print('blood var primer:', list(adb.var_names[:5]))
print('tumor var primer:', list(adt.var_names[:5]))
print('-> ce presek MAJHEN: join=inng izloci vecino -> prazne celice. To je vzrok.')

## KORAK B — replika notebooka 01: concat blood+tumor, preveri tumor po concat

In [ ]:
# Minimalna replika concat logike iz notebooka 01 cell-9
combined_tumor = anndata.concat([adt], join='inner', index_unique=None)
print('tumor sam po concat:', combined_tumor.shape,
      '| mediana countov:', np.median(np.asarray(combined_tumor.X.sum(axis=1)).ravel()))

# zdaj blood+tumor skupaj (kot cell-9: adata_all = concat([pbmc, tumor], join=inner))
both = anndata.concat([adb, adt], join='inner', index_unique=None)
print('blood+tumor concat:', both.shape)
n_blood = adb.shape[0]
sums_both = np.asarray(both.X.sum(axis=1)).ravel()
print('  blood del  mediana countov:', np.median(sums_both[:n_blood]))
print('  tumor del  mediana countov:', np.median(sums_both[n_blood:]))
print('  tumor praznih po concat:', int((sums_both[n_blood:]==0).sum()), '/', both.shape[0]-n_blood)
print('-> ce tumor mediana PADE po concat na ~0: join=inner je krivec.')

## KORAK C — barcode lookup (cell-11): se tumor celice pravilno povezejo?

Preveri ali Tissue_str lookup pravilno oznaci tumor celice (in ne izgubi countov).

In [ ]:
# Nalozi obstojeci data_rna + labels z Drive in preveri koncno stanje
import pickle
OUT = '/content/drive/MyDrive/Diploma/data/processed'
with open(os.path.join(OUT,'data_rna.pkl'),'rb') as fh: dr = pickle.load(fh)
with open(os.path.join(OUT,'data_labels.pkl'),'rb') as fh: dl = pickle.load(fh)
from scipy.sparse import issparse
sums_all = np.asarray(dr.sum(axis=1)).ravel() if issparse(dr) else dr.sum(axis=1)
# expm1 ni potreben za vsoto>0 test (log1p ohrani nicelnost)
tis = dl['Tissue'].values
for t,name in [(0,'blood'),(1,'tumor')]:
    m = tis==t
    s = sums_all[m]
    print(f'{name}: {m.sum()} celic | praznih(vsota=0): {int((s==0).sum())} | mediana ne-praznih: {np.median(s[s>0]) if (s>0).any() else 0:.2f}')
print('\n-> potrjuje ali so tumor celice prazne v KONCNEM data_rna (cez VSE paciente, ne le P24).')

## Zakljucek

Glede na to, kateri korak pokaze prazne celice:
- **KORAK A prazen** → tumorske .h5 so ze prazne (napacna datoteka? raw vs filtered? drug format?)
- **KORAK A2 majhen presek** → blood/tumor .h5 imata razlicna gene imena → join='inner' izprazni
- **KORAK B mediana pade** → concat(join='inner') je krivec → resitev: join='outer' + fillna(0), ali poravnaj gene pred concat
- **KORAK C** → potrdi obseg (vsi pacienti ali le nekateri)

Po najdbi vzroka: popravi notebook 01, regeneriraj data_rna, RE-TRENIRAJ TRIM.
